In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [24]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import pandas as pd
import time

# 🔹 Setup Chrome
options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless")  # optional

driver = webdriver.Chrome(options=options)

# 🔹 Open search page
driver.get("https://merojob.com/search?keywords=python")
time.sleep(8)

job_titles = []
companies = []
locations = []

page = 1

while True:
    print(f"\nScraping page {page}...")

    # 🔥 Scroll to load jobs
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

    # ✅ Job cards (your exact class)
    job_cards = driver.find_elements(
        By.XPATH,
        "//div[contains(@class,'my-4') and contains(@class,'break-words')]"
    )

    print(f"Found {len(job_cards)} jobs")

    for job in job_cards:
        # 🔹 Job Title
        try:
            title = job.find_element(By.XPATH, ".//h3").text
        except:
            title = "N/A"

        # 🔹 Company
        try:
            company = job.find_element(
                By.XPATH,
                ".//span[contains(@class,'text-left') and contains(@class,'line-clamp-1')]"
            ).text
        except:
            company = "N/A"

        # 🔹 Location
        try:
            location = job.find_element(
                By.XPATH,
                ".//span[contains(@class,'md:text-sm')]"
            ).text
        except:
            location = "N/A"

        job_titles.append(title)
        companies.append(company)
        locations.append(location)

    # 🔥 NEXT PAGE USING <a aria-label="Go to next page">
    try:
        next_btn = driver.find_element(
            By.XPATH,
            "//a[@aria-label='Go to next page']"
        )

        # Stop if disabled
        if next_btn.get_attribute("aria-disabled") == "true":
            print("Reached last page.")
            break

        # ✅ Better: navigate using href (more stable than click)
        next_url = next_btn.get_attribute("href")

        if not next_url:
            print("No next URL found. Stopping.")
            break

        driver.get(next_url)
        page += 1
        time.sleep(6)

    except:
        print("Next button not found. Stopping.")
        break

# 🔹 Close browser
driver.quit()

# 🔹 Create DataFrame
df = pd.DataFrame({
    "Job Title": job_titles,
    "Company": companies,
    "Location": locations
})

# 🔹 Clean duplicates
df.drop_duplicates(inplace=True)

# 🔹 Output
print(df.head())
print(f"\nTotal jobs scraped: {len(df)}")

# 🔹 Save CSV
df.to_csv("merojob_all_jobs.csv", index=False)
print("Saved to merojob_all_jobs.csv")


Scraping page 1...
Found 6 jobs

Scraping page 2...
Found 6 jobs

Scraping page 3...
Found 6 jobs

Scraping page 4...
Found 6 jobs

Scraping page 5...
Found 6 jobs

Scraping page 6...
Found 6 jobs

Scraping page 7...
Found 6 jobs

Scraping page 8...
Found 6 jobs

Scraping page 9...
Found 6 jobs

Scraping page 10...
Found 6 jobs

Scraping page 11...
Found 6 jobs

Scraping page 12...
Found 6 jobs

Scraping page 13...
Found 6 jobs

Scraping page 14...
Found 6 jobs

Scraping page 15...
Found 6 jobs

Scraping page 16...
Found 6 jobs

Scraping page 17...
Found 6 jobs

Scraping page 18...
Found 6 jobs

Scraping page 19...
Found 6 jobs

Scraping page 20...
Found 6 jobs

Scraping page 21...
Found 6 jobs

Scraping page 22...
Found 6 jobs

Scraping page 23...
Found 6 jobs

Scraping page 24...
Found 6 jobs

Scraping page 25...
Found 6 jobs

Scraping page 26...
Found 6 jobs

Scraping page 27...
Found 6 jobs

Scraping page 28...
Found 6 jobs

Scraping page 29...
Found 6 jobs

Scraping page 30...
Fo

Anti‑scraping defenses and rate limiting are two of the main roadblocks in modern web scraping and API‑based data collection. Both are designed to protect servers from abuse and to distinguish “real users” from bots, which makes robust scraping harder and more complex.

What rate limiting challenges look like
Rate limiting restricts how many requests a client (IP or account) can send in a given time window; when exceeded, services return 429 Too Many Requests errors or block the client. Common challenges include:

Tuning thresholds: choosing limits that stop scraping but don’t inconvenience legitimate traffic is hard, especially with mixed traffic patterns and APIs serving many clients.

Distributed systems: in multi‑server or cloud setups, each node must share rate‑limit state (e.g., Redis counters), which introduces latency and synchronization issues.

Performance overhead: every request check adds computation, and badly implemented rate‑limiting logic can itself become a bottleneck.

Typical anti‑scraping techniques
Beyond rate limiting, sites use a mix of client‑side and server‑side defenses to detect and block bots:

CAPTCHA and JavaScript challenges: captchas, image puzzles, or JS‑based “proof of work” force bots to act like human browsers, slowing down or blocking automated scrapers.

IP‑based blocking and geo‑restrictions: frequently rotating or geographically suspicious IPs are rate‑limited or banned, forcing scrapers to use proxies or rotating IPs.

Behavior and fingerprint analysis: servers track mouse movements, click patterns, TLS fingerprints, and browser characteristics; anomalies flag a client as a bot.

Dynamic content and hidden traps: content loaded via JS, lazy‑loaded elements, or “honeypot” links that only bots trigger silently trip detection systems.

How scrapers try to cope
To work around these, scrapers often:

Introduce delays and jitter between requests and use rotating IPs or proxy pools to avoid hitting per‑IP limits.

Use headless browsers (Puppeteer, Playwright) with randomized mouse movements, scroll patterns, and realistic navigation to mimic humans.

Rotate user‑agents, headers, and TLS fingerprints and switch configurations when detection is suspected.

Higher‑level design and ethical issues
From a system‑design perspective, rate‑limiting must balance security, fairness, and scalability, often using algorithms like token bucket, sliding‑window counters, or distributed rate‑limiting with shared state. On the scraping side, anti‑scraping arms races raise legal and ethical concerns: aggressive scraping can violate terms of service and data‑protection laws, so many organizations now prefer official APIs or licensed data partners instead of circumventing protections.

If you tell me whether you’re asking in the context of building a scraper, defending a web service, or system‑design homework, I can walk you through concrete patterns and code‑level best practices.